# P5b: FigQuant dequantization variants

This notebook runs the three approved variants:

1. Full FP32 dequantization (baseline)
2. Full BF16 dequantization (dtype control)
3. Output-row-tiled BF16 dequantization (memory variant)

The benchmark reports sampled process RSS peaks and the isolated dequantization RSS delta.

In [ ]:
# Set this to the repository URL or an existing mounted checkout.
REPO_URL = ""  # e.g. "https://github.com/ORG/REPO.git"
REPO_DIR = "/content/littlefig"

import os, subprocess, sys
if REPO_URL and not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
if os.path.exists(REPO_DIR):
    os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

In [ ]:
!pip -q install psutil
!python -m pip check

In [ ]:
# Fast validation with a small synthetic matrix; no model download.
!python benchmark/experiment_dequant_variants_v1.py --smoke --iterations 1 --results-path /content/p5b_smoke.json

In [ ]:
# Full P3a-shaped run. Increase/decrease iterations for runtime.
ITERATIONS = 20
BATCH_SIZE = 2
SEQUENCE_LENGTH = 256
TILE_ROWS = 128
RESULTS_PATH = "/content/p5b_dequant_variants_results.json"

import subprocess, os
cmd = [sys.executable, "benchmark/experiment_dequant_variants_v1.py", "--iterations", str(ITERATIONS), "--batch-size", str(BATCH_SIZE), "--sequence-length", str(SEQUENCE_LENGTH), "--tile-rows", str(TILE_ROWS), "--results-path", RESULTS_PATH]
subprocess.run(cmd, check=True)
assert os.path.exists(RESULTS_PATH), f"Benchmark finished without creating {RESULTS_PATH}"
print("Results written to", RESULTS_PATH)

In [ ]:
import json
from collections import defaultdict
if not os.path.exists(RESULTS_PATH):
    raise RuntimeError(f"Missing {RESULTS_PATH}. Run the full benchmark cell above first.")
with open(RESULTS_PATH, encoding="utf-8") as f:
    results = json.load(f)
summary = defaultdict(list)
for row in results["cases"]:
    summary[row["variant"]].append(row)
for variant, rows in summary.items():
    peak = max(r["rss_peak_mib"] for r in rows)
    delta = max(r["dequant_peak_delta_mib"] for r in rows)
    print(f"{variant}: peak RSS={peak:.1f} MiB, max dequant delta={delta:.1f} MiB")
print(json.dumps(results, indent=2))